# 02. Калибровка игровой экономики

**Задача.** Подобрать параметры экономики (`content/economy.json`) так, чтобы она обладала нужными свойствами (модель экономики v1, документ проекта `economy-model-v1`).

**Метод — вычислительный эксперимент.** Для каждой комбинации параметров прогоняем одних и тех же виртуальных детей (одинаковый seed — «общие случайные числа»: разница между вариантами возникает из-за параметров, а не из-за случайности) и проверяем шесть целевых свойств:

1. «50/30/20» копит на среднюю цель 4–6 недель;
2. «50/30/20» копит на маленькую цель 1–2 недели;
3. у «50/30/20» «особых желаний» не больше 0.1 в неделю;
4. у «50/30/20» на нужное уходит 45–60 % дохода;
5. «забывчивый» тратит на еду больше, чем «50/30/20» — **цена бездействия**;
6. «копилка» не скупает всё сразу: на большую цель — не быстрее 4 недель.

Эксперимент идёт от исходной экономики v1.0 (`ml/calibration/economy_v1_0.json`). Полный прогон ноутбука занимает несколько минут.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))  # чтобы импортировать генератор finny_sim
from finny_sim.calibrate import BASELINE, GRID, GRID_FINE, TARGETS, run_grid, summarize
from finny_sim.content import load_content
from finny_sim.simulate import run_simulation

PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
STRATEGIES = ['balanced', 'saver', 'spender', 'impulsive', 'forgetful']
STRATEGY_RU = {'balanced': '50/30/20', 'saver': 'копилка', 'spender': 'транжира',
               'impulsive': 'импульсивный', 'forgetful': 'забывчивый'}
KEYS = ['decay_per_day', 'goal_multiplier', 'nudge']
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e5e4df', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'lines.linewidth': 2})
print('Сетка этапа 1:', GRID)

## Этап 1. Грубая сетка

4 скорости голода × 3 уровня цен целей × подсказка «остаток в копилку» вкл/выкл = 24 варианта.

In [ ]:
stage1 = pd.DataFrame(run_grid(GRID, children=80, weeks=10, seed=2026))
configs1 = stage1.drop_duplicates(KEYS)
pd.pivot_table(configs1, index=['nudge', 'decay_per_day'], columns='goal_multiplier', values='targets_met')

Сколько из 6 целевых свойств выполнено (строки — подсказка и скорость голода, столбцы — множитель цен целей). Ни один вариант не выполняет все шесть.

In [ ]:
bal = stage1[stage1.strategy == 'balanced']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, nudge in zip(axes, [False, True]):
    ax.axhspan(4, 6, color='#e5e4df', zorder=0, label='цель: 4–6 недель')
    for color, mult in zip(PALETTE, GRID['goal_multiplier']):
        d = bal[(bal.nudge == nudge) & (bal.goal_multiplier == mult)].sort_values('decay_per_day')
        ax.plot(d.decay_per_day, d.weeks_to_medium.clip(upper=20), color=color, marker='o', markersize=5,
                label=f'цены ×{mult}')
    ax.set_title('подсказка включена' if nudge else 'подсказка выключена', loc='left')
    ax.set_xlabel('Голод, пунктов в день')
axes[0].set_ylabel('Недель накопления на среднюю цель')
axes[0].legend(frameon=False, loc='upper left')
fig.suptitle('«50/30/20»: сколько копить на среднюю цель', x=0.01, ha='left')
plt.tight_layout()

In [ ]:
wallet = bal.groupby('nudge').wallet_end.mean().rename({False: 'подсказка выкл.', True: 'подсказка вкл.'})
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(wallet.index, wallet.values, color=PALETTE[0], width=0.5)
for x, v in enumerate(wallet.values):
    ax.text(x, v + 3, f'{v:.0f}', ha='center')
ax.set_title('«50/30/20»: монет в кошельке к концу периода', loc='left')
ax.grid(axis='x', visible=False)
plt.tight_layout()

**Что показал этап 1.**
- Без подсказки «50/30/20» откладывает лишь ≈ 8 % дохода, а в кошельке залёживаются сотни монет. Подсказка в итогах недели решает это: остаток уходит в копилку, и доля накоплений вырастает до ≈ 30 %.
- С подсказкой цели при исходных ценах становятся слишком дешёвыми; множитель ×1.5 попадает в коридор 4–6 недель.
- **Неожиданное:** «забывчивый» почти во всех вариантах тратит на еду **меньше** заботливого. Пока питомец голоден, он ничего не «съедает» — время голода для кошелька бесплатно, а «особое желание» за 18 монет эту «экономию» не перекрывает. Игра нечаянно учит, что не кормить выгодно.

In [ ]:
sub = stage1[(stage1.goal_multiplier == 1.5) & stage1.nudge]
need = sub.pivot(index='decay_per_day', columns='strategy', values='need_per_week')
fig, ax = plt.subplots(figsize=(6, 3.5))
for color, strategy in zip(PALETTE, ['balanced', 'forgetful']):
    ax.plot(need.index, need[strategy], color=color, marker='o', markersize=5, label=STRATEGY_RU[strategy])
ax.set_xlabel('Голод, пунктов в день')
ax.set_ylabel('Траты на еду, монет в неделю')
ax.set_title('Цена бездействия при особом угощении за 18', loc='left')
ax.legend(frameon=False)
plt.tight_layout()

## Этап 2. Уточнение и цена «особого желания»

Около лучшего варианта (цены ×1.5, подсказка включена) перебираем скорость голода 16–18 и цену особого угощения 18, 30, 40.

In [ ]:
stage2 = pd.DataFrame(run_grid(GRID_FINE, children=80, weeks=10, seed=2026))
configs2 = stage2.drop_duplicates(KEYS + ['treat_price'])
pd.pivot_table(configs2, index='decay_per_day', columns='treat_price', values='targets_met')

In [ ]:
wide = stage2.pivot_table(index=['decay_per_day', 'treat_price'], columns='strategy',
                          values=['need_per_week', 'very_hungry_share'])
gap = (wide['need_per_week']['forgetful'] - wide['need_per_week']['balanced']).unstack(0)
impulsive_hungry = wide['very_hungry_share']['impulsive'].unstack(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
for color, decay in zip(PALETTE, gap.columns):
    ax1.plot(gap.index, gap[decay], color=color, marker='o', markersize=5, label=f'голод {decay}/день')
    ax2.plot(impulsive_hungry.index, impulsive_hungry[decay], color=color, marker='o', markersize=5,
             label=f'голод {decay}/день')
ax1.axhline(0, color='#888780', linewidth=1)
ax1.set_title('«Забывчивый» минус «50/30/20», еда за неделю', loc='left')
ax2.set_title('«Импульсивный»: доля времени очень голоден', loc='left')
ax2.yaxis.set_major_formatter(lambda v, _: f'{v:.0%}')
for ax in (ax1, ax2):
    ax.set_xlabel('Цена особого угощения')
ax1.legend(frameon=False)
plt.tight_layout()

**Решение — экономика v1.1:**
- голод **17** пунктов в день (на нужное у «50/30/20» ≈ 46 % дохода);
- цены целей **×1.5**: мороженое 45, мишка 70, машинка 105, велосипед 180, ноутбук 225 (обучающий колпак остаётся 15);
- подсказка «остаток в копилку» в итогах недели **включена**;
- «особое желание» — **праздничный торт за 30** (ассет `food/special/birthday-cake` уже есть в приложении).

Почему торт 30, а не 40: при 30 забота уже обходится дешевле бездействия (разница выше нуля), а при 40 «импульсивные» попадают в воронку — почти половину времени питомец очень голоден и денег не хватает ни на что. Это уже ловушка, а не урок.

Пятое свойство теперь выполняется, шестое тоже; единственное пограничное — «особых желаний» у «50/30/20» ≈ 0.11–0.12 в неделю (цель ≤ 0.1), то есть примерно одно за 8–9 недель. Это допустимо.

## До и после: v1.0 против v1.1

In [ ]:
economy_v10 = json.loads(BASELINE.read_text(encoding='utf-8'))
economy_v11 = load_content().economy  # текущий content/economy.json
before = pd.DataFrame(summarize(run_simulation(150, 10, 2026, economy=economy_v10), economy_v10)).T
after = pd.DataFrame(summarize(run_simulation(150, 10, 2026, economy=economy_v11), economy_v11)).T

cols = {'need_share': 'нужное', 'save_share': 'копилка', 'weeks_to_medium': 'недель на ср. цель',
        'wishes_per_week': 'особых желаний/нед', 'very_hungry_share': 'очень голоден', 'wallet_end': 'в кошельке'}
table = pd.concat({'v1.0': before[list(cols)], 'v1.1': after[list(cols)]}, names=['версия'])
table = table.rename(columns=cols).rename(index=STRATEGY_RU, level=1).astype(float).round(2)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
labels = {'need_share': 'нужное', 'want_share': 'желаемое', 'save_share': 'в копилку', 'rest': 'остаток'}
for ax, (title, frame) in zip(axes, [('v1.0 — до калибровки', before), ('v1.1 — после', after)]):
    share = frame.loc[STRATEGIES, ['need_share', 'want_share', 'save_share']].astype(float)
    share['rest'] = (1 - share.sum(axis=1)).clip(lower=0)
    left = pd.Series(0.0, index=share.index)
    for color, col in zip(PALETTE, share.columns):
        ax.barh(share.index.map(STRATEGY_RU), share[col], left=left, color=color,
                edgecolor='white', linewidth=2, height=0.6, label=labels[col])
        left += share[col]
    ax.set_xlim(0, 1)
    ax.set_title(title, loc='left')
    ax.grid(axis='y', visible=False)
axes[0].legend(frameon=False, ncol=4, loc='upper left', bbox_to_anchor=(0, -0.15))
plt.tight_layout()

## Выводы и ограничения

- Экономика v1.1 выполняет 5 из 6 целевых свойств; шестое пограничное.
- Подсказка в итогах недели — главный рычаг: без неё «правильная» стратегия 50/30/20 копит лишь 8 % дохода.
- Найден и устранён дефект дизайна: при дешёвом «особом желании» бездействие было выгоднее заботы.
- «Транжира» и «импульсивный» почти не достигают целей, но и не попадают в тупик — так и задумано: последствия видны, но выход есть всегда.

**Ограничения.** Параметры подобраны на синтетических детях, поведение которых — допущения. На пилоте фактические показатели (доля нужного, недели до цели, частота «особых желаний») нужно сравнить с этими и при необходимости повторить калибровку тем же ноутбуком.